# Engenharia de Features e Preparação de Dados - Datathon Passos Mágicos

## Introdução
Este notebook documenta o processo de transformação dos dados brutos em um dataset pronto para o treinamento de modelos de Machine Learning. O foco é criar variáveis que capturem nuances do desenvolvimento estudantil e definir uma variável alvo robusta para a predição de risco acadêmico.

**Etapas:**
1. Carregamento e Padronização de Indicadores.
2. Tratamento de Valores Ausentes.
3. Criação de Novas Features (Engenharia).
4. Definição da Variável Alvo (`risk_gap`).
5. Exportação do Dataset Processado.

## 1. Carregamento e Padronização

Iniciamos carregando o dataset original e padronizando os nomes das colunas para facilitar a manipulação.

In [ ]:
import pandas as pd
import numpy as np
import os

# Carregamento
df = pd.read_csv('../data/raw/PEDE_PASSOS_DATASET_FIAP.csv', sep=';')

# Seleção de Indicadores de 2020
indicadores = ['INDE_2020', 'IDA_2020', 'IEG_2020', 'IAA_2020', 'IPS_2020', 'IPP_2020', 'IAN_2020', 'IPV_2020']
df_proc = df[indicadores].copy()

# Padronização de Nomes
df_proc.columns = [col.split('_')[0] for col in df_proc.columns]

# Conversão Numérica
def to_numeric(col_data):
    return pd.to_numeric(col_data.astype(str).str.replace(',', '.'), errors='coerce')

for col in df_proc.columns:
    df_proc[col] = to_numeric(df_proc[col])

df_proc.head()

## 2. Tratamento de Valores Ausentes

Utilizamos a mediana para preencher valores nulos, garantindo que não percamos registros importantes e mantendo a distribuição estatística dos indicadores.

In [ ]:
# Preenchimento com a mediana
df_proc = df_proc.fillna(df_proc.median())
print(f'Valores nulos após tratamento: {df_proc.isnull().sum().sum()}')

## 3. Engenharia de Features

Criamos novas variáveis para enriquecer o modelo:
- **perception_gap**: Diferença entre autoavaliação (IAA) e desempenho real (IDA).
- **behavioral_score**: Média entre engajamento (IEG) e suporte psicossocial (IPS).
- **relative_performance**: Desempenho do aluno em relação à média da instituição.

In [ ]:
# Feature 1: Gap de Percepção
df_proc['perception_gap'] = df_proc['IAA'] - df_proc['IDA']

# Feature 2: Score Comportamental
df_proc['behavioral_score'] = (df_proc['IEG'] + df_proc['IPS']) / 2

# Feature 3: Performance Relativa
mean_ida = df_proc['IDA'].mean()
df_proc['relative_performance'] = df_proc['IDA'] / mean_ida

df_proc[['perception_gap', 'behavioral_score', 'relative_performance']].describe()

## 4. Definição da Variável Alvo

Definimos o `risk_gap` como nossa variável alvo binária. Alunos com **IAN < 6** são classificados como em risco (1), enquanto os demais são classificados como sem risco imediato (0).

In [ ]:
# Criando o Target
df_proc['risk_gap'] = (df_proc['IAN'] < 6).astype(int)

print('Distribuição do Target:')
print(df_proc['risk_gap'].value_counts(normalize=True))

## 5. Exportação do Dataset Processado

O dataset final é salvo para ser utilizado no treinamento dos modelos.

In [ ]:
output_path = '../data/processed/processed_data.csv'
os.makedirs(os.path.dirname(output_path), exist_ok=True)
df_proc.to_csv(output_path, index=False)
print(f'Dataset exportado com sucesso para: {output_path}')